In [1]:
import random
import torch
import os

import pandas as pd
import polars as pl
import numpy as np

import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder

import polars as pl
import pandas as pd

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [2]:


# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)



In [3]:


# ------------------ Metric Setup (experiment_config.py) ------------------
metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
	corpus_metrics.traditional_biber_distance,
	corpus_metrics.zero_wasserstein_distance
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in metrics]

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	elif metric in (corpus_metrics.traditional_biber_distance,corpus_metrics.zero_wasserstein_distance):
		c = corpus
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings



In [4]:
setA = ['can you tell me how i would normally say thank you as a french person', 'can you translate hi into spanish for me', 'can you translate milk into spanish for me', 'how can i say thank you very much in chinese', 'how can i thank somebody in italian', 'how could i say twin in chinese', 'how do germans say goodnight','how do i ask about the weather in chinese', 'how do i say hotel in finnish', 'how do i say bathroom in italian']
setB = ['how can i say thank you very much in chinese', 'how can i thank somebody in italian', 'how could i say twin in chinese', 'how do they say tacos in mexico', 'how do they say yes in brazil', 'how do vietnameses people say hello', 'how do you say cat in spanish', 'how do you say dog in spanish', 'how do you say fast in spanish', 'how do you say good bye in french', 'how do you say goodbye in spanish', 'how do you say hello in french', 'how do you say hello in japanese', 'how do you say hello in mexico']


In [5]:
def get_distances_from_compare_corpora(setA, setB):    
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    distances = {}
    for metric_name, metric in zip(metrics_names, metrics):
        if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
            tempA, tempB = tokensA, tokensB
        elif metric in (corpus_metrics.traditional_biber_distance,corpus_metrics.zero_wasserstein_distance):
            tempA, tempB = setA, setB
        else:
            tempA, tempB = embeddingsA, embeddingsB
        distances[metric_name] = metric(corpus1=tempA, corpus2=tempB)

    return distances

get_distances_from_compare_corpora(setA, setB)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 10 Num fake: 14
Num real: 10 Num fake: 14


WARNING clustering 24 points to 2 centroids: please provide at least 78 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_01_past_tense', 'f_02_perfect_aspect', 'f_04_place_adverbials', 'f_05_time_adverbials', 'f_09_pronoun_it', 'f_10_demonstrative_pronoun', 'f_12_proverb_do', 'f_15_gerunds', 'f_18_by_passives', 'f_19_be_main_verb', 'f_20_existential_there', 'f_21_that_verb_comp', 'f_22_that_adj_comp', 'f_23_wh_clause', 'f_24_infinitives', 'f_25_present_participle', 'f_26_past_participle', 'f_27_past_participle_whiz', 'f_28_present_participle_whiz', 'f_29_that_subj', 'f_30_that_obj', 'f_31_wh_subj', 'f_32_wh_obj', 'f_33_pied_piping', 'f_34_sentence_relatives', 'f_35_because', 'f_36_though', 'f_37_if', 'f_38_other_adv_sub', 'f_41_adj_pred', 'f_45_conjuncts', 'f_46_downtoners', 'f_47_hedges', '

{'CHI': np.float64(0.9987177546738071),
 'ZIPF': np.float64(0.11185366601975888),
 'CLASSIFIER': 0.4,
 'IRPR': np.float32(0.2202979),
 'FID': np.float64(0.7671713502935393),
 'PR': np.float64(0.05263102493103944),
 'DC': np.float64(0.0),
 'MAUVE': 0.09751853996329019,
 'TRADITIONAL': np.float64(0.6472167785842953),
 'ZERO': np.float64(0.15992175706619274)}